# Smile Detection CNN — live, in-browser demo

Runs entirely on this Binder session — no Google account, no install. A small sample of the GENKI-4K dataset (380 images) ships right in this repo under `data/`, so this notebook reads from disk instead of mounting Google Drive.

Run each cell in order (Shift+Enter). Training and evaluation should finish in a few minutes.

In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout
import PIL
import matplotlib.pyplot as plt

In [ ]:
import pathlib
path_to_train = "data/train_images"
path_to_test = "data/test_images"

train_genki = pathlib.Path(path_to_train).with_suffix('')
test_genki = pathlib.Path(path_to_test).with_suffix('')

In [ ]:
train_genki

In [ ]:
image_count = len(list(train_genki.glob('*/*.jpg')))
print(image_count)

In [ ]:
image_count = len(list(test_genki.glob('*/*.jpg')))
print(image_count)

In [ ]:
smiling = list(train_genki.glob('smiling/*'))
PIL.Image.open(str(smiling[0]))

In [ ]:
not_smiling = list(train_genki.glob('not_smiling/*'))
PIL.Image.open(str(not_smiling[0]))

## Image size

GENKI-4K images are small face crops, so 128×128 is used.

In [ ]:
batch_size = 32
img_height = 128
img_width = 128

In [ ]:
train_genki, validation_genki = tf.keras.utils.image_dataset_from_directory(
  train_genki,
  validation_split=0.2,
  subset="both",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)

In [ ]:
class_names = train_genki.class_names
print(class_names)

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_genki.take(1):
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[labels[i]])
    plt.axis("off")

In [ ]:
for image_batch, labels_batch in train_genki:
  print(image_batch.shape)
  print(labels_batch.shape)
  break

In [ ]:
#image_batch.numpy()

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_genki = train_genki.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
validation_genki = validation_genki.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
normalization_layer = tf.keras.layers.Rescaling(1./255)

In [ ]:
normalized_images = train_genki.map(lambda x, y: (normalization_layer(x), y))
image_batch, labels_batch = next(iter(normalized_images))
first_image = image_batch[0]
print(np.min(first_image), np.max(first_image))

In [ ]:
layers = tf.keras.layers

data_augmentation = keras.Sequential(
  [
    layers.RandomFlip("horizontal",
                      input_shape=(img_height,
                                  img_width,
                                  3)),
    layers.RandomRotation(0.02),
  ]
)

In [ ]:
plt.figure(figsize=(10, 10))
for images, _ in train_genki.take(1):
  for i in range(9):
    augmented_images = data_augmentation(images)
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(augmented_images[0].numpy().astype("uint8"))
    plt.axis("off")

In [ ]:
num_classes = len(class_names)

model = Sequential([
  data_augmentation,
  layers.Rescaling(1./255, input_shape=(img_height, img_width, 3)),
  layers.Conv2D(16, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(32, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(64, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Dropout(0.2),
  layers.Flatten(),
  layers.Dense(128, activation='relu'),
  layers.Dense(num_classes)
])

In [ ]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [ ]:
epochs=5
history = model.fit(
  train_genki,
  validation_data = validation_genki,
  epochs=epochs
)

In [ ]:
train_genki.take(1)

In [ ]:
model.metrics_names

In [ ]:
acc = history.history['accuracy']
validation_acc = history.history['val_accuracy']

loss = history.history['loss']
validation_loss = history.history['val_loss']

epochs_range = range(epochs)

plt.figure(figsize=(8, 8))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, validation_acc, label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, validation_loss, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

## Evaluate on the test set


In [ ]:
import os

accurate = 0
test_cases = 0

path_to_test1 = "data/test_images/smiling"
test1 = pathlib.Path(path_to_test1).with_suffix('')

folder = test1

true_pos = 0
true_neg = 0
false_pos = 0
false_neg = 0

y_true = []
y_pred = []

for filename in os.listdir(test1):
  f = os.path.join(folder, filename)
  print(f)
  img = tf.keras.utils.load_img(
      f, target_size=(img_height, img_width)
  )
  img_array = tf.keras.utils.img_to_array(img)
  img_array = tf.expand_dims(img_array, 0) # Create a batch

  predictions = model.predict(img_array)
  score = tf.nn.softmax(predictions[0])
  pred_class = class_names[np.argmax(score)]

  print(
      "This image most likely belongs to {} with a {:.2f} percent confidence."
      .format(pred_class, 100 * np.max(score))
  )

  test_cases = test_cases + 1
  if pred_class == 'smiling':
      true_pos = true_pos + 1
  else:
    false_neg = false_neg + 1

print("********************************")
print("********************************")

path_to_test2 = "data/test_images/not_smiling"
test2 = pathlib.Path(path_to_test2).with_suffix('')

folder = test2

for filename in os.listdir(folder):
  f = os.path.join(folder, filename)
  print(f)
  img = tf.keras.utils.load_img(
      f, target_size=(img_height, img_width)
  )
  img_array = tf.keras.utils.img_to_array(img)
  img_array = tf.expand_dims(img_array, 0) # Create a batch

  predictions = model.predict(img_array)
  score = tf.nn.softmax(predictions[0])
  pred_class = class_names[np.argmax(score)]

  print(
      "This image most likely belongs to {} with a {:.2f} percent confidence."
      .format(class_names[np.argmax(score)], 100 * np.max(score))
  )

  test_cases = test_cases + 1
  print(pred_class)
  if pred_class == 'not_smiling':
      true_neg = true_neg + 1
  else:
    false_pos = false_pos + 1

print("********************************")
print("********************************")

accuracy = (true_pos + true_neg) / test_cases

precision = true_pos / (false_pos + true_pos)
recall = true_pos / (false_neg + true_pos)

print("number of correct predictions:", (true_pos + true_neg))
print("accuracy:", accuracy)

print("precision:", precision)
print("recall:", recall)

print(true_pos, true_neg, false_pos, false_neg)

In [ ]:
# Find the model's last convolutional layer -- Grad-CAM reads what it detected
last_conv_layer_name = [layer.name for layer in model.layers if 'conv2d' in layer.name][-1]
print("Last convolutional layer:", last_conv_layer_name)

In [ ]:
# Pick one test image to explain -- change this path to try a different image
example_path = os.path.join(path_to_test1, os.listdir(path_to_test1)[0])

img = tf.keras.utils.load_img(example_path, target_size=(img_height, img_width))
img_array = tf.keras.utils.img_to_array(img)
img_array = tf.expand_dims(img_array, 0)  # add a batch dimension

plt.imshow(img_array[0].numpy().astype("uint8"))
plt.title("Original image")
plt.axis("off")

In [ ]:
# A version of the model that also reports the last conv layer's output.
# Rebuilt from a fresh Input so Keras traces every layer (including the
# nested data_augmentation block) -- same trained weights, just a graph
# Grad-CAM can read from.
inputs = keras.Input(shape=(img_height, img_width, 3))
x = inputs
for layer in model.layers:
    x = layer(x)
    if layer.name == last_conv_layer_name:
        conv_output = x

grad_model = tf.keras.models.Model(inputs, [conv_output, x])

In [ ]:
# Watch how much each part of that layer contributed to the predicted class
with tf.GradientTape() as tape:
    conv_output, predictions = grad_model(img_array)
    predicted_class = tf.argmax(predictions[0])
    class_score = predictions[:, predicted_class]

grads = tape.gradient(class_score, conv_output)
pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

heatmap = conv_output[0] @ pooled_grads[..., tf.newaxis]
heatmap = tf.squeeze(heatmap)
heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)

plt.imshow(heatmap)
plt.title("Raw heatmap")

In [ ]:
# Overlay the heatmap on the original image
plt.imshow(img_array[0].numpy().astype("uint8"))
plt.imshow(heatmap, cmap="jet", alpha=0.4, extent=(0, img_width, img_height, 0))
plt.title(f"Grad-CAM -- predicted: {class_names[predicted_class]}")
plt.axis("off")
plt.show()